# Notebook 09 — Final Global Comparison

This notebook is the final comparison point of the experimental pipeline. It does **not** train new models. It loads the frozen outputs from the previous notebooks and builds one unified benchmark.

**Inputs expected:**

- N03 GRU baseline outputs
- N04 LSTM baseline outputs
- N05 Transformer baseline outputs
- N08 classical ML baseline outputs

**Main metric:** present-class Macro-F1. This remains the primary metric because some client test sets do not contain all five water-quality classes.

**Supporting metric added here:** mean ordinal class-distance error. This metric is calculated from saved test predictions where prediction files are available. It is used only for interpretation, not for model ranking.

## Section 0 — Setup and source discovery

The notebook first creates a clean output folder and searches for the current result files. It can read files either from extracted Kaggle datasets or directly from attached `.zip` outputs.

In [1]:
# 0.1 — Imports and output folders
import os, io, zipfile, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", None)

ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
PRIMARY = "Macro_F1_present"

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUTPUT_DIR = WORKING / "final_comparison"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = OUTPUT_DIR / "tables"; TAB_DIR.mkdir(exist_ok=True)

FAMILY_COLORS = {
    "Classical ML": "#2E8B57",
    "Deep learning": "#1F3A5F",
    "Federated learning": "#C0392B",
}

print("Output folder:", OUTPUT_DIR)

Output folder: /kaggle/working/final_comparison


In [2]:
# 0.2 — Build a resource index for normal files and files inside zip archives
ROOTS = []
for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data")]:
    if p.exists():
        ROOTS.append(p)

normal_index = {}
zip_index = {}

for root in ROOTS:
    for dirpath, _, files in os.walk(root):
        for fn in files:
            p = Path(dirpath) / fn
            normal_index.setdefault(fn, []).append(p)
            if p.suffix.lower() == ".zip":
                try:
                    with zipfile.ZipFile(p) as z:
                        for name in z.namelist():
                            if not name.endswith("/"):
                                zip_index.setdefault(Path(name).name, []).append((p, name))
                except Exception:
                    pass


def find_resource(filename, prefer=None):
    candidates = []
    for p in normal_index.get(filename, []):
        candidates.append(("file", p, None, str(p)))
    for zp, member in zip_index.get(filename, []):
        candidates.append(("zip", zp, member, f"{zp}!{member}"))
    if prefer:
        preferred = [c for c in candidates if prefer.lower() in c[3].lower()]
        if preferred:
            candidates = preferred
    return candidates[0] if candidates else None


def resource_desc(res):
    return "MISSING" if res is None else res[3]


def read_csv_resource(filename, prefer=None):
    res = find_resource(filename, prefer=prefer)
    if res is None:
        raise FileNotFoundError(filename)
    typ, p, member, _ = res
    if typ == "file":
        return pd.read_csv(p)
    with zipfile.ZipFile(p) as z:
        with z.open(member) as f:
            return pd.read_csv(f)


def load_npz_resource(filename, prefer=None):
    res = find_resource(filename, prefer=prefer)
    if res is None:
        return None
    typ, p, member, _ = res
    if typ == "file":
        return np.load(p)
    with zipfile.ZipFile(p) as z:
        with z.open(member) as f:
            return np.load(io.BytesIO(f.read()))

required = {
    "gru_local": "local_test_summary.csv",
    "gru_central": "centralized_test_summary.csv",
    "gru_fed": "federated_all_variants_test_results.csv",
    "lstm_local": "local_lstm_test_summary.csv",
    "lstm_central": "centralized_lstm_test_summary.csv",
    "lstm_fed": "federated_lstm_all_variants_test_results.csv",
    "tx_local": "local_transformer_test_summary.csv",
    "tx_central": "centralized_transformer_test_summary.csv",
    "tx_fed": "federated_transformer_all_variants_test_results.csv",
    "ml_local": "local_classical_ml_test_results_ranked.csv",
    "ml_central": "centralized_classical_ml_test_results_ranked.csv",
}

prefer_map = {
    "gru_local": "gru", "gru_central": "gru", "gru_fed": "gru",
    "lstm_local": "lstm", "lstm_central": "lstm", "lstm_fed": "lstm",
    "tx_local": "transformer", "tx_central": "transformer", "tx_fed": "transformer",
    "ml_local": "classical", "ml_central": "classical",
}

located = {}
missing = []
for key, marker in required.items():
    res = find_resource(marker, prefer=prefer_map.get(key))
    located[key] = res
    print(f"{'✓' if res else '✗'} {key:<12} {marker:<55} {resource_desc(res)}")
    if res is None:
        missing.append(key)

assert not missing, f"Missing result files: {missing}. Attach current N03/N04/N05/N08 outputs."
print("\nAll required source files were found.")

✓ gru_local    local_test_summary.csv                                  /kaggle/input/datasets/hebacheblal/gru-baselines/audit/local_test_summary.csv
✓ gru_central  centralized_test_summary.csv                            /kaggle/input/datasets/hebacheblal/gru-baselines/audit/centralized_test_summary.csv
✓ gru_fed      federated_all_variants_test_results.csv                 /kaggle/input/datasets/hebacheblal/gru-baselines/audit/federated_all_variants_test_results.csv
✓ lstm_local   local_lstm_test_summary.csv                             /kaggle/input/datasets/hebacheblal/lstm-baselines/audit/local_lstm_test_summary.csv
✓ lstm_central centralized_lstm_test_summary.csv                       /kaggle/input/datasets/hebacheblal/lstm-baselines/audit/centralized_lstm_test_summary.csv
✓ lstm_fed     federated_lstm_all_variants_test_results.csv            /kaggle/input/datasets/hebacheblal/lstm-baselines/audit/federated_lstm_all_variants_test_results.csv
✓ tx_local     local_transformer_test_summ

## Section 1 — Assemble the unified result table

Each model contributes one row per client. Classical ML and non-federated deep-learning models are loaded from their frozen summary files. For federated learning, **all saved tested variants** are loaded rather than only one representative variant. This makes the final federated gap and best-federated-per-client analysis consistent with the confusion-matrix notebook and with the saved prediction files. The primary ranking still uses present-class Macro-F1.


In [3]:
# 1.1 — Helper functions for metrics and ordinal class-distance error

def mean_ordinal_from_npz(filename, prefer=None):
    data = load_npz_resource(filename, prefer=prefer)
    if data is None:
        return np.nan
    return float(np.mean(np.abs(data["y_true"].astype(int) - data["y_pred"].astype(int))))


def mean_ordinal_from_csv(filename, prefer=None):
    try:
        df = read_csv_resource(filename, prefer=prefer)
    except FileNotFoundError:
        return np.nan
    return float(np.mean(np.abs(df["y_true"].astype(int) - df["y_pred"].astype(int))))


def add_result_row(family, strategy, model, client, accuracy, balanced, macro_present, weighted, ordinal_error=np.nan):
    return {
        "Family": family,
        "Strategy": strategy,
        "Model": model,
        "Client": client,
        "Accuracy": accuracy,
        "Balanced_accuracy": balanced,
        "Macro_F1_present": macro_present,
        "Weighted_F1": weighted,
        "Mean_ordinal_class_error": ordinal_error,
    }


def load_dl_summary(summary_file, model, strategy, pred_pattern, prefer):
    df = read_csv_resource(summary_file, prefer=prefer)
    rows = []
    for _, r in df.iterrows():
        client = r["Client"]
        pred_file = pred_pattern.format(client=client, client_lower=client.lower())
        rows.append(add_result_row(
            "Deep learning", strategy, model, client,
            r["Accuracy"], r["Balanced_accuracy"], r["Macro_F1_present_classes"], r["Weighted_F1"],
            mean_ordinal_from_npz(pred_file, prefer=prefer)
        ))
    return rows


def load_dl_federated_all(summary_file, arch, pred_pattern, prefer):
    """Load all federated variants from a frozen federated result table.

    Earlier versions of N09 kept only one representative variant per architecture.
    For the final consistency pass, all variants are included so that the per-client
    best federated model and the confusion-matrix notebook use the same candidate set.
    """
    df = read_csv_resource(summary_file, prefer=prefer)
    rows = []
    for _, r in df.iterrows():
        client = r["Client"]
        variant = r["Variant"]
        pred_file = pred_pattern.format(variant=str(variant).lower(), client=client, client_lower=client.lower())
        rows.append(add_result_row(
            "Federated learning", "Federated", f"{arch} ({variant})", client,
            r["Accuracy"], r["Balanced_accuracy"], r["Macro_F1_present_classes"], r["Weighted_F1"],
            mean_ordinal_from_npz(pred_file, prefer=prefer)
        ))
    variant_means = df.groupby("Variant")["Macro_F1_present_classes"].mean().sort_values(ascending=False)
    return rows, variant_means


def load_ml(summary_file, prefer):
    df = read_csv_resource(summary_file, prefer=prefer)
    rows = []
    for _, r in df.iterrows():
        model = r["Model"]
        strategy = r["Strategy"]
        client = r["Eval_client"]
        model_file = str(model).lower()
        if strategy == "Local":
            pred_file = f"local_{model_file}_{client.lower()}_to_{client.lower()}_test_predictions.csv"
        else:
            pred_file = f"centralized_{model_file}_pooled_to_{client.lower()}_test_predictions.csv"
        rows.append(add_result_row(
            "Classical ML", strategy, model, client,
            r["Accuracy"], r["Balanced_accuracy"], r["Macro_F1_present_classes"], r["Weighted_F1"],
            mean_ordinal_from_csv(pred_file, prefer=prefer)
        ))
    return rows



In [4]:
# 1.2 — Build the unified long table
rows = []
rows += load_dl_summary("local_test_summary.csv", "GRU", "Local", "local_{client_lower}_test_preds.npz", "gru")
rows += load_dl_summary("centralized_test_summary.csv", "GRU", "Centralized", "centralized_{client_lower}_test_preds.npz", "gru")
gru_fed_rows, gru_variant_means = load_dl_federated_all("federated_all_variants_test_results.csv", "GRU", "{variant}_{client_lower}_test_preds.npz", "gru")
rows += gru_fed_rows

rows += load_dl_summary("local_lstm_test_summary.csv", "LSTM", "Local", "local_lstm_{client_lower}_test_preds.npz", "lstm")
rows += load_dl_summary("centralized_lstm_test_summary.csv", "LSTM", "Centralized", "centralized_lstm_{client_lower}_test_preds.npz", "lstm")
lstm_fed_rows, lstm_variant_means = load_dl_federated_all("federated_lstm_all_variants_test_results.csv", "LSTM", "lstm_{variant}_{client_lower}_test_preds.npz", "lstm")
rows += lstm_fed_rows

rows += load_dl_summary("local_transformer_test_summary.csv", "Transformer", "Local", "local_transformer_{client_lower}_test_preds.npz", "transformer")
rows += load_dl_summary("centralized_transformer_test_summary.csv", "Transformer", "Centralized", "centralized_transformer_{client_lower}_test_preds.npz", "transformer")
tx_fed_rows, tx_variant_means = load_dl_federated_all("federated_transformer_all_variants_test_results.csv", "Transformer", "transformer_{variant}_{client_lower}_test_preds.npz", "transformer")
rows += tx_fed_rows

rows += load_ml("local_classical_ml_test_results_ranked.csv", "classical")
rows += load_ml("centralized_classical_ml_test_results_ranked.csv", "classical")

long_df = pd.DataFrame(rows)
long_df.to_csv(TAB_DIR / "unified_results_long.csv", index=False)

print("Unified table rows:", len(long_df))
print("Model configurations:", long_df.groupby(["Family", "Strategy", "Model"]).ngroups)
print("Federated variants loaded and mean present-class Macro-F1:")
print("\nGRU:")
print(gru_variant_means.round(4).to_string())
print("\nLSTM:")
print(lstm_variant_means.round(4).to_string())
print("\nTransformer:")
print(tx_variant_means.round(4).to_string())

missing_ordinal = long_df[long_df["Mean_ordinal_class_error"].isna()]
if not missing_ordinal.empty:
    print("\nNote: ordinal error is unavailable for rows whose prediction files were not saved/found:")
    display(missing_ordinal[["Family", "Strategy", "Model", "Client"]])

display(long_df.head(12).round(4))



Unified table rows: 96
Model configurations: 32
Federated variants loaded and mean present-class Macro-F1:

GRU:
Variant
FedAvg_v3     0.4183
FedProx_v3    0.3934
FedProx_v2    0.2849
FedProx_v1    0.2809
FedAvg_v2     0.2273
FedAvg_v1     0.2185

LSTM:
Variant
FedProx_v1    0.4326
FedAvg_v3     0.4240
FedProx_v2    0.4204
FedProx_v3    0.3792
FedAvg_v2     0.2643
FedAvg_v1     0.2280

Transformer:
Variant
FedProx_v1    0.3509
FedProx_v2    0.2642
FedAvg_v2     0.2387
FedAvg_v1     0.2214
FedAvg_v3     0.2183
FedProx_v3    0.1969

Note: ordinal error is unavailable for rows whose prediction files were not saved/found:


,Family,Strategy,Model,Client
6,Federated learning,Federated,GRU (FedAvg_v1),England
7,Federated learning,Federated,GRU (FedAvg_v1),Ireland
8,Federated learning,Federated,GRU (FedAvg_v1),USA
9,Federated learning,Federated,GRU (FedAvg_v2),England
10,Federated learning,Federated,GRU (FedAvg_v2),Ireland
11,Federated learning,Federated,GRU (FedAvg_v2),USA
15,Federated learning,Federated,GRU (FedProx_v1),England
16,Federated learning,Federated,GRU (FedProx_v1),Ireland
17,Federated learning,Federated,GRU (FedProx_v1),USA
18,Federated learning,Federated,GRU (FedProx_v2),England


,Family,Strategy,Model,Client,Accuracy,Balanced_accuracy,Macro_F1_present,Weighted_F1,Mean_ordinal_class_error
0,Deep learning,Local,GRU,England,0.7988,0.6804,0.6941,0.8004,0.2073
1,Deep learning,Local,GRU,Ireland,0.8503,0.5445,0.4864,0.8675,0.1503
2,Deep learning,Local,GRU,USA,0.9186,0.7933,0.7909,0.9203,0.0834
3,Deep learning,Centralized,GRU,England,0.8001,0.6677,0.6910,0.8023,0.2088
4,Deep learning,Centralized,GRU,Ireland,0.8916,0.5285,0.5130,0.8965,0.1182
5,Deep learning,Centralized,GRU,USA,0.9207,0.7329,0.7718,0.9218,0.0935
6,Federated learning,Federated,GRU (FedAvg_v1),England,0.6435,0.4447,0.4870,0.6277,NaN
7,Federated learning,Federated,GRU (FedAvg_v1),Ireland,0.0846,0.4227,0.0672,0.0692,NaN
8,Federated learning,Federated,GRU (FedAvg_v1),USA,0.0960,0.1979,0.1014,0.1231,NaN
9,Federated learning,Federated,GRU (FedAvg_v2),England,0.6715,0.4457,0.4915,0.6501,NaN


## Section 2 — Main ranking by present-class Macro-F1

This is the primary ranking table. The best model is the one with the highest mean present-class Macro-F1 across England, Ireland, and USA.

In [5]:
# 2.1 — Unified ranking table
pivot = long_df.pivot_table(
    index=["Family", "Strategy", "Model"],
    columns="Client",
    values=PRIMARY,
    aggfunc="first"
).reindex(columns=ACTIVE_CLIENTS)
pivot["Mean"] = pivot[ACTIVE_CLIENTS].mean(axis=1)
pivot = pivot.sort_values("Mean", ascending=False)

ranking_out = pivot.reset_index().round(4)
ranking_out.to_csv(TAB_DIR / "unified_ranking_macro_f1_present.csv", index=False)
print("Unified ranking — primary metric: present-class Macro-F1")
display(ranking_out)

Unified ranking — primary metric: present-class Macro-F1


Client,Family,Strategy,Model,England,Ireland,USA,Mean
0,Classical ML,Local,XGBoost,0.7145,0.4995,0.7889,0.6676
1,Deep learning,Local,Transformer,0.7091,0.4955,0.7896,0.6647
2,Classical ML,Centralized,XGBoost,0.6699,0.5381,0.7735,0.6605
3,Deep learning,Centralized,GRU,0.6910,0.5130,0.7718,0.6586
4,Classical ML,Local,CatBoost,0.6973,0.4933,0.7848,0.6585
5,Deep learning,Centralized,Transformer,0.6719,0.5206,0.7810,0.6578
6,Deep learning,Local,GRU,0.6941,0.4864,0.7909,0.6571
7,Classical ML,Local,Random_Forest,0.6832,0.5030,0.7792,0.6551
8,Deep learning,Local,LSTM,0.6735,0.4907,0.7739,0.6460
9,Deep learning,Centralized,LSTM,0.6664,0.5142,0.7559,0.6455


In [6]:
# 2.2 — Full metrics by client
full_tbl = long_df.sort_values(["Client", PRIMARY], ascending=[True, False]).round(4)
full_tbl.to_csv(TAB_DIR / "unified_full_metrics_by_client.csv", index=False)
print("Full metric table saved. Preview for England:")
display(full_tbl[full_tbl["Client"] == "England"].reset_index(drop=True))

Full metric table saved. Preview for England:


,Family,Strategy,Model,Client,Accuracy,Balanced_accuracy,Macro_F1_present,Weighted_F1,Mean_ordinal_class_error
0,Classical ML,Local,XGBoost,England,0.8180,0.7016,0.7145,0.8190,0.1880
1,Deep learning,Local,Transformer,England,0.8122,0.7122,0.7091,0.8133,0.1939
2,Classical ML,Local,CatBoost,England,0.8087,0.6728,0.6973,0.8096,0.1965
3,Deep learning,Local,GRU,England,0.7988,0.6804,0.6941,0.8004,0.2073
4,Deep learning,Centralized,GRU,England,0.8001,0.6677,0.6910,0.8023,0.2088
5,Classical ML,Local,Random_Forest,England,0.8261,0.6472,0.6832,0.8240,0.1811
6,Deep learning,Local,LSTM,England,0.8007,0.6552,0.6735,0.8026,0.2052
7,Deep learning,Centralized,Transformer,England,0.8057,0.6597,0.6719,0.8081,0.2026
8,Classical ML,Centralized,XGBoost,England,0.8017,0.6595,0.6699,0.8050,0.2040
9,Deep learning,Centralized,LSTM,England,0.7845,0.6433,0.6664,0.7879,0.2232


## Section 3 — Ranking figure

This figure summarizes the benchmark visually. It should be used in Chapter 4 as the global comparison figure.

In [7]:
# 3.1 — Ranking bar chart
plot_df = pivot.reset_index().sort_values("Mean", ascending=True)
labels = [f"{r.Model} · {r.Strategy}" for r in plot_df.itertuples()]
colors = [FAMILY_COLORS[f] for f in plot_df["Family"]]

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(range(len(plot_df)), plot_df["Mean"], color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(labels, fontsize=8.5)
ax.set_xlabel("Mean present-class Macro-F1 across clients")
ax.set_title("Final global comparison — all models ranked", fontsize=12, fontweight="bold")
ax.set_xlim(0, max(0.80, float(plot_df["Mean"].max()) + 0.08))
for i, v in enumerate(plot_df["Mean"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=7.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=f) for f, c in FAMILY_COLORS.items()], loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "unified_ranking.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", FIG_DIR / "unified_ranking.png")

Saved: /kaggle/working/final_comparison/figures/unified_ranking.png


## Section 4 — Federated positioning

This section compares the best federated model with the best non-federated model for each client. This is the most direct evidence for whether federated learning helps in the current Stage 1 setting.

In [8]:
# 4.1 — Federated gap per client
# This uses all loaded federated variants, not only one representative variant per architecture.
gap_rows = []
best_fed_rows = []
for client in ACTIVE_CLIENTS:
    sub = long_df[long_df["Client"] == client]
    nonfed = sub[sub["Strategy"] != "Federated"]
    fed = sub[sub["Strategy"] == "Federated"]
    bnf = nonfed.loc[nonfed[PRIMARY].idxmax()]
    bf = fed.loc[fed[PRIMARY].idxmax()]
    gap_rows.append({
        "Client": client,
        "Best_non_federated": f"{bnf['Model']} ({bnf['Strategy']})",
        "Best_non_fed_F1": bnf[PRIMARY],
        "Best_federated": bf["Model"],
        "Best_fed_F1": bf[PRIMARY],
        "FL_gap": bf[PRIMARY] - bnf[PRIMARY],
    })
    best_fed_rows.append({
        "Client": client,
        "Best_federated_model": bf["Model"],
        "Accuracy": bf["Accuracy"],
        "Balanced_accuracy": bf["Balanced_accuracy"],
        "Macro_F1_present": bf[PRIMARY],
        "Weighted_F1": bf["Weighted_F1"],
        "Mean_ordinal_class_error": bf["Mean_ordinal_class_error"],
    })

gap_df = pd.DataFrame(gap_rows).round(4)
gap_df.to_csv(TAB_DIR / "federated_gap_per_client.csv", index=False)
print("Federated gap per client:")
display(gap_df)

best_fed_df = pd.DataFrame(best_fed_rows).round(4)
best_fed_df.to_csv(TAB_DIR / "best_federated_model_per_client.csv", index=False)
print("\nBest federated model per client using all loaded federated variants:")
display(best_fed_df)



Federated gap per client:


,Client,Best_non_federated,Best_non_fed_F1,Best_federated,Best_fed_F1,FL_gap
0,England,XGBoost (Local),0.7145,GRU (FedAvg_v2),0.4915,-0.2230
1,Ireland,XGBoost (Centralized),0.5381,LSTM (FedProx_v1),0.4294,-0.1087
2,USA,GRU (Local),0.7909,LSTM (FedAvg_v3),0.6785,-0.1124



Best federated model per client using all loaded federated variants:


,Client,Best_federated_model,Accuracy,Balanced_accuracy,Macro_F1_present,Weighted_F1,Mean_ordinal_class_error
0,England,GRU (FedAvg_v2),0.6715,0.4457,0.4915,0.6501,NaN
1,Ireland,LSTM (FedProx_v1),0.7049,0.4814,0.4294,0.7585,0.3190
2,USA,LSTM (FedAvg_v3),0.8969,0.6475,0.6785,0.8947,0.1208


In [9]:
# 4.2 — Federated gap figure
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(ACTIVE_CLIENTS))
w = 0.38
nf = gap_df["Best_non_fed_F1"].values
ff = gap_df["Best_fed_F1"].values
ax.bar(x - w/2, nf, w, label="Best non-federated", color="#1F3A5F")
ax.bar(x + w/2, ff, w, label="Best federated", color="#C0392B")
for i, (a, b) in enumerate(zip(nf, ff)):
    ax.text(i - w/2, a + 0.01, f"{a:.3f}", ha="center", fontsize=8)
    ax.text(i + w/2, b + 0.01, f"{b:.3f}", ha="center", fontsize=8)
    ax.annotate(f"{b-a:+.3f}", xy=(i, max(a, b) + 0.06), ha="center", fontsize=9, fontweight="bold", color="#C0392B")
ax.set_xticks(x)
ax.set_xticklabels(ACTIVE_CLIENTS)
ax.set_ylabel("Present-class Macro-F1")
ax.set_ylim(0, 0.95)
ax.set_title("Best federated vs best non-federated model", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "federated_gap.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", FIG_DIR / "federated_gap.png")

Saved: /kaggle/working/final_comparison/figures/federated_gap.png


## Section 5 — Classical ML vs deep learning

This section checks whether deep learning is truly necessary, or whether classical ML remains competitive. This comparison strengthens the thesis because it avoids claiming that neural models are better without evidence.

In [10]:
# 5.1 — Best classical ML vs best deep learning per client and strategy
h2h_rows = []
for client in ACTIVE_CLIENTS:
    for strategy in ["Local", "Centralized"]:
        sub = long_df[(long_df["Client"] == client) & (long_df["Strategy"] == strategy)]
        ml = sub[sub["Family"] == "Classical ML"]
        dl = sub[sub["Family"] == "Deep learning"]
        if ml.empty or dl.empty:
            continue
        bml = ml.loc[ml[PRIMARY].idxmax()]
        bdl = dl.loc[dl[PRIMARY].idxmax()]
        h2h_rows.append({
            "Client": client,
            "Strategy": strategy,
            "Best_ML": bml["Model"],
            "ML_F1": bml[PRIMARY],
            "Best_DL": bdl["Model"],
            "DL_F1": bdl[PRIMARY],
            "ML_minus_DL": bml[PRIMARY] - bdl[PRIMARY],
            "Winner": "Classical ML" if bml[PRIMARY] >= bdl[PRIMARY] else "Deep learning",
        })

h2h_df = pd.DataFrame(h2h_rows).round(4)
h2h_df.to_csv(TAB_DIR / "classical_ml_vs_dl_head_to_head.csv", index=False)
print("Classical ML vs deep learning:")
display(h2h_df)

Classical ML vs deep learning:


,Client,Strategy,Best_ML,ML_F1,Best_DL,DL_F1,ML_minus_DL,Winner
0,England,Local,XGBoost,0.7145,Transformer,0.7091,0.0054,Classical ML
1,England,Centralized,XGBoost,0.6699,GRU,0.6910,-0.0211,Deep learning
2,Ireland,Local,Random_Forest,0.5030,Transformer,0.4955,0.0075,Classical ML
3,Ireland,Centralized,XGBoost,0.5381,Transformer,0.5206,0.0175,Classical ML
4,USA,Local,XGBoost,0.7889,GRU,0.7909,-0.0020,Deep learning
5,USA,Centralized,XGBoost,0.7735,Transformer,0.7810,-0.0075,Deep learning


## Section 6 — Best model per client

The best model can differ by client. This table is more useful than forcing a single model to be declared best everywhere.

In [11]:
# 6.1 — Best and runner-up model per client
best_rows = []
for client in ACTIVE_CLIENTS:
    sub = long_df[long_df["Client"] == client].sort_values(PRIMARY, ascending=False)
    b = sub.iloc[0]
    runner = sub.iloc[1]
    best_rows.append({
        "Client": client,
        "Best_model": f"{b['Model']} ({b['Strategy']}, {b['Family']})",
        "Best_F1": b[PRIMARY],
        "Best_ordinal_error": b["Mean_ordinal_class_error"],
        "Runner_up": f"{runner['Model']} ({runner['Strategy']})",
        "Runner_F1": runner[PRIMARY],
        "Runner_ordinal_error": runner["Mean_ordinal_class_error"],
    })

best_df = pd.DataFrame(best_rows).round(4)
best_df.to_csv(TAB_DIR / "best_model_per_client_overall.csv", index=False)
print("Best model per client:")
display(best_df)

Best model per client:


,Client,Best_model,Best_F1,Best_ordinal_error,Runner_up,Runner_F1,Runner_ordinal_error
0,England,"XGBoost (Local, Classical ML)",0.7145,0.1880,Transformer (Local),0.7091,0.1939
1,Ireland,"XGBoost (Centralized, Classical ML)",0.5381,0.0938,Transformer (Centralized),0.5206,0.1106
2,USA,"GRU (Local, Deep learning)",0.7909,0.0834,Transformer (Local),0.7896,0.0778


## Section 7 — Supporting ordinal class-distance error

Water-quality classes are ordered. A wrong prediction from **Good** to **Excellent** is less severe than a wrong prediction from **Poor** to **Excellent**. For this reason, this notebook adds the mean ordinal class-distance error as a supporting interpretation metric.

This metric is not used to rank models. The main ranking remains present-class Macro-F1.

In [12]:
# 7.1 — Ordinal class-distance error table
ordinal_pivot = long_df.pivot_table(
    index=["Family", "Strategy", "Model"],
    columns="Client",
    values="Mean_ordinal_class_error",
    aggfunc="first"
).reindex(columns=ACTIVE_CLIENTS)
ordinal_pivot["Mean"] = ordinal_pivot[ACTIVE_CLIENTS].mean(axis=1)
ordinal_pivot = ordinal_pivot.sort_values("Mean", ascending=True)

ordinal_out = ordinal_pivot.reset_index().round(4)
ordinal_out.to_csv(TAB_DIR / "ordinal_class_distance_error_by_model.csv", index=False)
print("Supporting metric — lower is better:")
display(ordinal_out)

# A compact table for the top Macro-F1 models only
ranked_index = pivot.head(8).index
top_ordinal = ordinal_pivot.reindex(ranked_index).reset_index().round(4)
top_ordinal.to_csv(TAB_DIR / "ordinal_error_for_top_macro_f1_models.csv", index=False)
print("\nOrdinal error for top Macro-F1 models:")
display(top_ordinal)

Supporting metric — lower is better:


Client,Family,Strategy,Model,England,Ireland,USA,Mean
0,Classical ML,Local,Random_Forest,0.1811,0.0837,0.0662,0.1104
1,Classical ML,Centralized,Random_Forest,0.1867,0.0863,0.0753,0.1161
2,Classical ML,Centralized,XGBoost,0.2040,0.0938,0.0844,0.1274
3,Deep learning,Centralized,Transformer,0.2026,0.1106,0.0808,0.1313
4,Deep learning,Local,Transformer,0.1939,0.1373,0.0778,0.1363
5,Classical ML,Centralized,CatBoost,0.2047,0.1012,0.1046,0.1369
6,Classical ML,Local,XGBoost,0.1880,0.1361,0.0904,0.1382
7,Deep learning,Centralized,GRU,0.2088,0.1182,0.0935,0.1402
8,Deep learning,Centralized,LSTM,0.2232,0.1106,0.0930,0.1423
9,Classical ML,Local,CatBoost,0.1965,0.1447,0.0950,0.1454



Ordinal error for top Macro-F1 models:


Client,Family,Strategy,Model,England,Ireland,USA,Mean
0,Classical ML,Local,XGBoost,0.1880,0.1361,0.0904,0.1382
1,Deep learning,Local,Transformer,0.1939,0.1373,0.0778,0.1363
2,Classical ML,Centralized,XGBoost,0.2040,0.0938,0.0844,0.1274
3,Deep learning,Centralized,GRU,0.2088,0.1182,0.0935,0.1402
4,Classical ML,Local,CatBoost,0.1965,0.1447,0.0950,0.1454
5,Deep learning,Centralized,Transformer,0.2026,0.1106,0.0808,0.1313
6,Deep learning,Local,GRU,0.2073,0.1503,0.0834,0.1470
7,Classical ML,Local,Random_Forest,0.1811,0.0837,0.0662,0.1104


## Section 8 — Final synthesis

This final reading is automatically based on the current loaded result files. It should be used as the basis for the Chapter 4 global-comparison discussion.

In [13]:
# 8.1 — Automated synthesis
nonfed_mean = pivot[pivot.index.get_level_values("Strategy") != "Federated"]["Mean"]
fed_mean = pivot[pivot.index.get_level_values("Strategy") == "Federated"]["Mean"]

top_nonfed = nonfed_mean.max()
best_fed = fed_mean.max()
top5_spread = nonfed_mean.nlargest(5).max() - nonfed_mean.nlargest(5).min()
ml_wins = int((h2h_df["Winner"] == "Classical ML").sum())
all_h2h = len(h2h_df)

print("=" * 72)
print("NOTEBOOK 09 — FINAL SYNTHESIS")
print("=" * 72)
print(f"""
1. The best non-federated models are close to each other.
   The top five non-federated configurations span only {top5_spread:.3f} present-class Macro-F1.
   This means model family alone is not the main limitation.

2. Classical ML is competitive with deep learning.
   Classical ML wins {ml_wins}/{all_h2h} client-strategy comparisons, and the margins are small.
   This justifies including XGBoost/Random Forest/CatBoost as serious baselines.

3. Federated learning remains below the best non-federated models.
   Best federated mean Macro-F1 = {best_fed:.3f}, while best non-federated mean Macro-F1 = {top_nonfed:.3f}.
   The per-client gap table uses all loaded federated variants, so it gives the fairest available federated comparison.

4. The main bottleneck is likely data-related.
   The results point to client heterogeneity, class imbalance, and modal/repeated-value patterns as stronger limitations
   than the choice between GRU, LSTM, Transformer, or classical ML.

5. Mean ordinal class-distance error is reported only as a supporting metric.
   The primary ranking remains present-class Macro-F1.
""")



NOTEBOOK 09 — FINAL SYNTHESIS

1. The best non-federated models are close to each other.
   The top five non-federated configurations span only 0.009 present-class Macro-F1.
   This means model family alone is not the main limitation.

2. Classical ML is competitive with deep learning.
   Classical ML wins 3/6 client-strategy comparisons, and the margins are small.
   This justifies including XGBoost/Random Forest/CatBoost as serious baselines.

3. Federated learning remains below the best non-federated models.
   Best federated mean Macro-F1 = 0.433, while best non-federated mean Macro-F1 = 0.668.
   The per-client gap table uses all loaded federated variants, so it gives the fairest available federated comparison.

4. The main bottleneck is likely data-related.
   The results point to client heterogeneity, class imbalance, and modal/repeated-value patterns as stronger limitations
   than the choice between GRU, LSTM, Transformer, or classical ML.

5. Mean ordinal class-distance error

In [14]:
# 9 — Zip outputs
zip_path = shutil.make_archive(str(WORKING / "final_comparison"), "zip", root_dir=OUTPUT_DIR)
print("Archive created:", zip_path)
print("\nFiles produced:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUT_DIR))

Archive created: /kaggle/working/final_comparison.zip

Files produced:
  figures/federated_gap.png
  figures/unified_ranking.png
  tables/best_federated_model_per_client.csv
  tables/best_model_per_client_overall.csv
  tables/classical_ml_vs_dl_head_to_head.csv
  tables/federated_gap_per_client.csv
  tables/ordinal_class_distance_error_by_model.csv
  tables/ordinal_error_for_top_macro_f1_models.csv
  tables/unified_full_metrics_by_client.csv
  tables/unified_ranking_macro_f1_present.csv
  tables/unified_results_long.csv
